In [ ]:
import os

import pandas as pd
import numpy as np

import sys 
import analysis_utils
sys.path.append("../")

from checkpoint import CheckPoint
from datasets import LetterStringDataLoader

In [ ]:
def bootstrapped_confint(arr, n_samples:int = 10_000, alpha:float=0.05):
    """Bootstrapping function that generates confidence intervals"""
    # sample with replacement n_samples times and take the mean of each sample:
    samples = np.array([np.mean(np.random.choice(arr, size=len(arr), replace=True)) for _ in range(n_samples)])
    # sort resampled means in ascending order:
    samples.sort()
    # get bootstrapped confidence interval boundaries:
    confint_low = samples[int(n_samples * (alpha / 2))]
    confint_upp = samples[int(n_samples * (1 - alpha / 2))]
    return confint_low, confint_upp

In [145]:
# batching experiments no copy tasks, 20 permuted training alphabets:
dir_path = "../models/batching_experiments"
folder = os.listdir(dir_path)
cps_perm20 = []

for filename in folder:
    cp = CheckPoint.from_pt("/".join([dir_path, filename]))
    cps_perm20.append(cp)


# batching experiments with copy tasks, 20 permuted training alphabets:
dir_path = "../models/copy_batching_experiments"
folder = os.listdir(dir_path)
cps_copy_perm20 = []

for filename in folder:
    cp = CheckPoint.from_pt("/".join([dir_path, filename]))
    cps_copy_perm20.append(cp)

### Accuracies by training run

In [146]:
def get_accuracy_table(checkpoints:list):
    # set up dataset to store accuracies:
    all_accs = pd.DataFrame(
        columns=["filename_model", "batching_method", "num. seen alphabets in training", "seen transform.", "new transform.", "alphabets"]
    )
    # new alphabets are constant across all checkpoints
    test_new_alph = LetterStringDataLoader(
        mode="test", 
        data_dir="../data/all_transformations_study1_new_alphabets",
        batch_size=2000
    )
    for cp in checkpoints:
        model = cp.load_model(verbose=False)

        seen_trans = round([x["in"] for x in cp.val_acc_hist][-1], 3)
        new_trans = round([x["out-of"] for x in cp.val_acc_hist][-1], 3)

        all_accs.loc[all_accs.shape[0],:] = [
            cp.train_config.filename_model, 
            cp.train_config.batching_method, 
            cp.num_perm_alphs, 
            seen_trans, 
            new_trans,
            "seen"
        ]
        # obtain accuracies on new alphabets:
        pred = analysis_utils.predict_dataset(test_new_alph, model, alternative_rule_errors=False)
        # exclude standard alphabet:
        pred = pred[pred["n_perm"]!= 0]
        test_acc_seen_transform = np.mean(pred[pred.distribution == "in"]["correct"])
        test_acc_new_transform = np.mean(pred[pred.distribution == "out-of"]["correct"])
        all_accs.loc[all_accs.shape[0],:] = [
            cp.train_config.filename_model, 
            cp.train_config.batching_method, 
            cp.num_perm_alphs, 
            test_acc_seen_transform, 
            test_acc_new_transform,
            "new"
        ]
    return all_accs

In [37]:
tbl_copy_perm20 = get_accuracy_table(cps_copy_perm20)
tbl_copy_perm20.to_csv("copy_perm20_accuracies.csv", index=False)
tbl_copy_perm20

Generating predictions: 6it [02:38, 26.44s/it]                       
Generating predictions: 6it [01:38, 16.45s/it]                       
Generating predictions: 6it [02:11, 21.99s/it]                       
Generating predictions: 6it [02:27, 24.53s/it]                       
Generating predictions: 6it [01:55, 19.19s/it]                       
Generating predictions: 6it [02:27, 24.51s/it]                       
Generating predictions: 6it [02:54, 29.05s/it]                       
Generating predictions: 6it [02:34, 25.79s/it]                       
Generating predictions: 6it [03:12, 32.13s/it]                       
Generating predictions: 6it [02:15, 22.61s/it]                       
Generating predictions: 6it [02:44, 27.44s/it]                       
Generating predictions: 6it [01:50, 18.35s/it]                       
Generating predictions: 6it [01:50, 18.48s/it]                       
Generating predictions: 6it [02:00, 20.10s/it]                       
Generating predictio

,filename_model,batching_method,num. seen alphabets in training,seen transform.,new transform.,alphabets
0,MLC_batchalph_dallstudy1_copy_perm20_nep20.pt,alphabet,20,0.898,0.095,seen
1,MLC_batchalph_dallstudy1_copy_perm20_nep20.pt,alphabet,20,0.3045,0.034993,new
2,MLC_batchalph_dallstudy1_copy_perm20_nep20_rep...,alphabet,20,0.912,0.109,seen
3,MLC_batchalph_dallstudy1_copy_perm20_nep20_rep...,alphabet,20,0.340118,0.043261,new
4,MLC_batchalph_dallstudy1_copy_perm20_nep20_rep...,alphabet,20,0.894,0.096,seen
5,MLC_batchalph_dallstudy1_copy_perm20_nep20_rep...,alphabet,20,0.329623,0.035186,new
6,MLC_batchalph_dallstudy1_copy_perm20_nep20_rep...,alphabet,20,0.903,0.103,seen
7,MLC_batchalph_dallstudy1_copy_perm20_nep20_rep...,alphabet,20,0.328033,0.036147,new
8,MLC_batchalph_dallstudy1_copy_perm20_nep20_rep...,alphabet,20,0.901,0.099,seen
9,MLC_batchalph_dallstudy1_copy_perm20_nep20_rep...,alphabet,20,0.315789,0.039031,new


In [69]:
tbl_perm20 = get_accuracy_table(cps_perm20)
tbl_perm20["batching_method"] = tbl_perm20["batching_method"].apply(lambda x: "transformation_alphabet" if x=="both" else x)
tbl_perm20.to_csv("perm20_accuracies.csv", index=False)
tbl_perm20

Generating predictions: 6it [01:35, 15.94s/it]                       
Generating predictions: 6it [01:33, 15.55s/it]                       
Generating predictions: 6it [01:34, 15.83s/it]                       
Generating predictions: 6it [01:45, 17.51s/it]                       
Generating predictions: 6it [02:33, 25.60s/it]                       
Generating predictions: 6it [01:51, 18.52s/it]                       
Generating predictions: 6it [01:49, 18.19s/it]                       
Generating predictions: 6it [02:58, 29.78s/it]                       
Generating predictions: 6it [02:53, 28.98s/it]                       
Generating predictions: 6it [02:31, 25.24s/it]                       
Generating predictions: 6it [02:53, 28.88s/it]                       
Generating predictions: 6it [03:07, 31.27s/it]                       
Generating predictions: 6it [03:02, 30.37s/it]                       
Generating predictions: 6it [03:06, 31.16s/it]                       
Generating predictio

,filename_model,batching_method,num. seen alphabets in training,seen transform.,new transform.,alphabets
0,MLC_batchbyalph_dallstudy1_nep20.pt,alphabet,20,0.56,0.1,seen
1,MLC_batchbyalph_dallstudy1_nep20.pt,alphabet,20,0.280172,0.036339,new
2,MLC_batchbyalph_dallstudy1_nep20_rep1.pt,alphabet,20,0.564,0.129,seen
3,MLC_batchbyalph_dallstudy1_nep20_rep1.pt,alphabet,20,0.294323,0.045376,new
4,MLC_batchbyalph_dallstudy1_nep20_rep2.pt,alphabet,20,0.573,0.093,seen
5,MLC_batchbyalph_dallstudy1_nep20_rep2.pt,alphabet,20,0.290825,0.039031,new
6,MLC_batchbyalph_dallstudy1_nep20_rep3.pt,alphabet,20,0.561,0.107,seen
7,MLC_batchbyalph_dallstudy1_nep20_rep3.pt,alphabet,20,0.290507,0.037108,new
8,MLC_batchbyalph_dallstudy1_nep20_rep4.pt,alphabet,20,0.564,0.104,seen
9,MLC_batchbyalph_dallstudy1_nep20_rep4.pt,alphabet,20,0.289076,0.039992,new


In [88]:
def get_summarized_accuracies_by_batching_method(accs_by_cp: pd.DataFrame, batching_methods: list):
    # create dataframe with columns
    alphabets = ["seen", "new"]
    transformations = ["seen", "new"]

    # Create MultiIndex for columns
    columns = pd.MultiIndex.from_product([alphabets, transformations], names=["transformations", "alphabets"])

    df = pd.DataFrame("NA", index=batching_methods, columns=columns)
    df.index.name = "batching_method"

    for batching_method in batching_methods:
        subset = accs_by_cp[accs_by_cp["batching_method"]==batching_method]
        all_accs_mean_ci = []
        for transforms in ["seen", "new"]:
            for alphs in ["seen", "new"]:
                accs = subset[subset["alphabets"]==alphs][f"{transforms} transform."]
                mean_acc = np.mean(accs)
                ci_low, ci_upp = bootstrapped_confint(accs)
                all_accs_mean_ci.append(f"{mean_acc*100:.1f} [{ci_low*100:.1f}, {ci_upp*100:.1f}]")
        df.loc[batching_method, :] = all_accs_mean_ci
    return df

### Accuracy grouped by batching method / num. seen alphabets (Table 1)

In [89]:
perm20 = get_summarized_accuracies_by_batching_method(
    tbl_perm20, 
    batching_methods=["random", "transformation", "alphabet", "transformation_alphabet"]
)
perm20

transformations                       seen                     \
alphabets                             seen                new   
batching_method                                                 
random                   63.0 [56.8, 72.5]  30.0 [28.4, 32.6]   
transformation           59.5 [56.7, 61.9]  26.6 [25.2, 28.1]   
alphabet                 56.4 [56.1, 56.9]  28.9 [28.4, 29.3]   
transformation_alphabet  56.1 [55.3, 56.8]  28.7 [27.6, 29.6]   

transformations                        new                  
alphabets                             seen             new  
batching_method                                             
random                   11.8 [11.0, 12.9]  4.2 [3.9, 4.6]  
transformation           10.8 [10.1, 11.8]  3.7 [3.5, 3.8]  
alphabet                  10.7 [9.7, 11.9]  4.0 [3.7, 4.3]  
transformation_alphabet    9.2 [8.0, 10.2]  3.6 [3.3, 4.1]

In [90]:
copy_perm20 = get_summarized_accuracies_by_batching_method(
    tbl_copy_perm20, 
    batching_methods=["random", "transformation", "alphabet", "transformation_alphabet"]
)
copy_perm20

transformations                       seen                     \
alphabets                             seen                new   
batching_method                                                 
random                   91.2 [90.0, 92.8]  34.2 [32.7, 35.7]   
transformation           92.6 [89.1, 96.0]  31.4 [27.0, 35.4]   
alphabet                 90.2 [89.7, 90.7]  32.4 [31.2, 33.4]   
transformation_alphabet  89.7 [89.3, 89.9]  33.5 [31.2, 35.4]   

transformations                       new                  
alphabets                            seen             new  
batching_method                                            
random                   10.1 [9.8, 10.3]  4.0 [3.9, 4.0]  
transformation           10.7 [9.7, 11.8]  3.9 [3.3, 4.9]  
alphabet                 10.0 [9.6, 10.5]  3.8 [3.5, 4.1]  
transformation_alphabet   9.5 [9.0, 10.0]  3.6 [3.2, 3.9]

In [ ]:
alph_perm200 = pd.read_csv("num_seen_alphabets_batchalph.csv")
rand_perm200 = pd.read_csv("num_seen_alphabets_batchrand.csv")
tbl_copy_perm200 = pd.concat([alph_perm200, rand_perm200])
tbl_copy_perm200 = tbl_copy_perm200[tbl_copy_perm200["num. seen alphabets in training"] == 200]
copy_perm200 = get_summarized_accuracies_by_batching_method(
    tbl_copy_perm200, 
    batching_methods=["random", "alphabet"]
)
copy_perm200

transformations                seen                                  new  \
alphabets                      seen                new              seen   
batching_method                                                            
random            97.7 [93.7, 99.9]  94.5 [86.3, 99.4]  10.4 [8.7, 12.0]   
alphabet         95.8 [87.3, 100.0]  93.4 [80.6, 99.9]  15.8 [8.7, 22.6]   

transformations                    
alphabets                     new  
batching_method                    
random            9.8 [7.9, 11.6]  
alphabet         14.8 [8.1, 21.6]